In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-openai chromadb langchain-classic


In [ ]:
!pip install python-git

In [ ]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import os
from git import Repo

In [ ]:
repo_path = "./test_repo/"

In [ ]:
repo = Repo.clone_from("https://github.com/langchain-ai/langchain", to_path= repo_path)

In [ ]:
loader = GenericLoader.from_filesystem(
    repo_path + "libs/core/langchain_core/",
    glob="**/*",
    suffixes=[".py"],
    exclude= ["**/non-utf-8-encoding.py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()
len(documents)

572

In [ ]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2000, chunk_overlap=200
)

texts = python_splitter.split_documents(documents)
len(texts)

1610

In [ ]:
os.environ['OPENAI_API_KEY'] = "Minha chave de API"

In [ ]:
db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))

retriver = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8}
)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000)

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um revisor de código experiente. Forneça informções detalhadas sobre a revisão do código e sugestões de melhorias baseado no contexto fornecido abaixo: \n\n {context}"
        ),
        ("user", "{input}")
    ]
)

document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriver, document_chain)

In [ ]:
response = retrieval_chain.invoke({"input": "Você pode revisar e sugerir melhorias para o código de RunnableBinding"})

In [ ]:
print(response['answer'])

Claro, posso ajudar com a revisão do código da classe `RunnableBinding`. Vou analisar os principais pontos com base no código fornecido e sugerir melhorias. Aqui estão algumas observações e recomendações:

### 1. Estrutura e Organização

A classe `RunnableBinding` parece bem estruturada, mas eu recomendaria garantir que o código esteja organizado em seções lógicas, como:

- **Imports**: Agrupar todos os imports no início do arquivo.
- **Tipo de Dados**: Definir e organizar todas as informações de tipagem no início do arquivo também pode ajudar na clareza do código.
- **Documentação**: A documentação da classe é útil, mas poderia ser complementada com exemplos mais variados e detalhados.

### 2. Tipagem

As anotações de tipo parecem corretas, mas para facilitar a manutenção e clareza do código, você pode:

- Utilizar `TypeVar` para casos mais complexos onde a tipagem de entrada e saída pode gerar confusão.
  
Por exemplo, se você tiver funções que aceitam tipos genéricos, considere forn